# MedVision-AI: Kaggle GPU Cloud Environment & Execution Pipeline

This notebook provides the Cloud GPU Host Environment for MedVision-AI.
Source Repository: https://github.com/SwastikPandey1024/MedVision-AI

MLOps Strategy: Local laptop executes fast CPU smoke tests. This Kaggle notebook executes Phase 1 EDA, Phase 2 Data Pipelines, and Phase 3/4 Cloud GPU Training Verification under MirroredStrategy.

In [ ]:
# Cell 1: Environment & Dynamic Hardware Verification Script
import os
import sys
import platform

os.environ["KERAS_BACKEND"] = "tensorflow"
import tensorflow as tf
import keras

print("=" * 75)
print("MedVision-AI Kaggle GPU Environment Report")
print("=" * 75)
print(f"Python Version       : {platform.python_version()} ({sys.executable})")
print(f"TensorFlow Version   : {tf.__version__}")
print(f"Keras Version        : {keras.__version__}")
print(f"Keras Backend        : {os.environ.get('KERAS_BACKEND', 'tensorflow')}")

gpus = tf.config.list_physical_devices('GPU')
gpu_count = len(gpus)
print(f"\nGPU Availability     : {'YES (GPU Enabled)' if gpu_count > 0 else 'NO (CPU Fallback)'}")
print(f"GPU Device Count     : {gpu_count}")

if gpu_count > 0:
    for i, gpu in enumerate(gpus):
        print(f"\n--- GPU Device #{i+1} Details ---")
        print(f"Device Identifier    : {gpu.name}")
        try:
            details = tf.config.experimental.get_device_details(gpu)
            print(f"Hardware Model       : {details.get('device_name', 'NVIDIA GPU')}")
        except Exception as err:
            print(f"Hardware Details     : {err}")
else:
    print("\n[NOTE] No GPU device discovered in current Kaggle kernel session.")
print("=" * 75)

In [ ]:
# Cell 2: Robust Repository Setup & Package Installation
!rm -rf /kaggle/working/MedVision-AI
!git clone https://github.com/SwastikPandey1024/MedVision-AI.git /kaggle/working/MedVision-AI
%cd /kaggle/working/MedVision-AI
!git fetch origin
!git reset --hard origin/main
!pip install -e . --no-build-isolation

# Verification of Working Directory & Package Import
!pwd
!ls scripts/train.py
!python -c "import medvision; print('MedVision package import: OK')"

In [ ]:
# Cell 3: Phase 1 — RSNA Dataset Ingestion & Data Engineering
%cd /kaggle/working/MedVision-AI
from medvision.data.dataset import find_dataset_root, parse_rsna_manifest
from medvision.data.validation import validate_manifest_integrity
from medvision.data.splits import create_patient_aware_splits, verify_zero_patient_leakage
from medvision.data.eda import generate_eda_report

print("=" * 75)
print("Phase 1: RSNA Dataset Ingestion & Data Engineering")
print("=" * 75)
ds_root = find_dataset_root()
print(f"[1] Dataset Root Resolved: {ds_root}")
manifest_df = parse_rsna_manifest(ds_root)
print(f"[2] Manifest Parsed: {len(manifest_df)} unique patient records.")
val_audit = validate_manifest_integrity(manifest_df)
print(f"[3] Data Quality Audit Valid: {val_audit['is_valid']}")
train_df, val_df, test_df = create_patient_aware_splits(manifest_df, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15)
leakage_audit = verify_zero_patient_leakage(train_df, val_df, test_df)
print(f"[5] Patient Leakage Audit: Zero Leakage = {leakage_audit['has_zero_leakage']}")
print("=" * 75)

In [ ]:
# Cell 4: Phase 3/4 — Kaggle Cloud GPU Smoke Test Execution
# Executes latest repository script `scripts/train.py` under MirroredStrategy
# Target Environment: 2 x Tesla T4 GPUs, MirroredStrategy (replicas=2), FP16, 4 train steps, 2 val steps
%cd /kaggle/working/MedVision-AI
!python scripts/train.py --mode dev --smoke-test --mixed-precision --architecture densenet121